# Data Reconciliation after Post-Feature-Engineering Cleaning

## Purpose

Use the group-approved `data_reclean_after_fe.csv` **without regenerating or replacing the agreed cleaning work**.

The notebook only reconciles derived features whose source columns were changed by the post-feature-engineering cleaning step. It preserves all other columns exactly, validates the data contract, records lineage, and produces the single model-development input:

- `data_reconcile/artifacts/data_reconciled_after_fe.csv`
- `data_reconcile/artifacts/reconciliation_metadata.json`
- `data_reconcile/reports/reconciliation_audit.csv`

The two outcome helpers, `final_result_encoded` and `passed`, are retained for profiling but explicitly excluded from clustering.


In [1]:
import hashlib
import json
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

KEY_COLUMNS = ["id_student", "code_module", "code_presentation"]
SOURCE_FILENAME = "data_reclean_after_fe.csv"

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
OUTPUT_ROOT = WORK_ROOT / "data_reconcile"
ARTIFACT_DIR = OUTPUT_ROOT / "artifacts"
REPORT_DIR = OUTPUT_ROOT / "reports"

if OUTPUT_ROOT.exists():
    shutil.rmtree(OUTPUT_ROOT)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)


def discover_unique_file(filename):
    roots = [INPUT_ROOT, Path.cwd(), Path("/mnt/data")]
    matches = []
    for root in roots:
        if root.exists():
            matches.extend(path for path in root.rglob(filename) if path.is_file())
    matches = sorted(set(path.resolve() for path in matches))
    if not matches:
        raise FileNotFoundError(
            f"Could not find {filename}. Attach it as a Kaggle input."
        )
    if len(matches) > 1:
        raise RuntimeError(
            f"Found multiple copies of {filename}:\n"
            + "\n".join(str(path) for path in matches)
            + "\nAttach only the intended source."
        )
    return matches[0]


source_path = discover_unique_file(SOURCE_FILENAME)
print("Source:", source_path)
print("Output root:", OUTPUT_ROOT)


Source: /kaggle/input/datasets/bachtruonggia/edtech/data_reclean_after_fe.csv
Output root: /kaggle/working/data_reconcile


## 1. Load and validate the agreed recleaned dataset

The notebook treats the input file as immutable. It requires one row per learner-course-presentation key, validates the outcome mappings, and refuses missing or infinite numeric values.


In [2]:
source_df = pd.read_csv(source_path)
source_df.columns = source_df.columns.astype(str)

required_columns = {
    *KEY_COLUMNS,
    "final_result",
    "final_result_encoded",
    "passed",
    "login_weekly",
    "video_completion_rate",
    "forum_posts_count",
    "days_to_first_filled",
    "submission_timeliness_days",
    "frequency_score",
    "forum_per_login",
    "completion_x_frequency",
    "low_engagement_flag",
    "engagement_breadth",
    "vle_total_clicks",
    "vle_active_days",
    "vle_distinct_sites",
    "log_vle_total_clicks",
    "log_vle_active_days",
    "log_vle_distinct_sites",
    "clicks_per_active_day",
    "log_clicks_per_active_day",
}
missing = sorted(required_columns.difference(source_df.columns))
if missing:
    raise ValueError(f"Missing required columns: {missing}")

if source_df.duplicated(KEY_COLUMNS).any():
    duplicate_count = int(source_df.duplicated(KEY_COLUMNS, keep=False).sum())
    raise ValueError(f"Duplicate learner-course-presentation rows: {duplicate_count}")

if source_df[KEY_COLUMNS].isna().any().any():
    raise ValueError("Key columns contain missing values")

numeric_columns = source_df.select_dtypes(include=[np.number]).columns.tolist()
numeric_values = source_df[numeric_columns].to_numpy(dtype=float)
if not np.isfinite(numeric_values).all():
    raise ValueError("Numeric columns contain missing or infinite values")

expected_encoded = source_df["final_result"].map({
    "Withdrawn": 0,
    "Fail": 1,
    "Pass": 2,
    "Distinction": 3,
})
if expected_encoded.isna().any():
    unknown = sorted(source_df.loc[expected_encoded.isna(), "final_result"].unique())
    raise ValueError(f"Unknown final_result values: {unknown}")
if not np.array_equal(expected_encoded.astype(int), source_df["final_result_encoded"].astype(int)):
    raise ValueError("final_result_encoded does not match the agreed mapping")

expected_passed = source_df["final_result"].isin(["Pass", "Distinction"]).astype(int)
if not np.array_equal(expected_passed, source_df["passed"].astype(int)):
    raise ValueError("passed does not match Pass/Distinction = 1")

with source_path.open("rb") as file:
    source_sha256 = hashlib.file_digest(file, "sha256").hexdigest()

print("Shape:", source_df.shape)
print("Unique keys:", len(source_df))
print("SHA256:", source_sha256)
display(source_df.head())


Shape: (32593, 53)
Unique keys: 32593
SHA256: 37feae077762afc954f46cae8e9c2d6c9882a5d7ce874c35d4b9835b1ccbe757


,id_student,code_module,code_presentation,login_weekly,video_completion_rate,forum_posts_count,days_to_first_filled,submission_timeliness_days,never_started,missed_submission,late_submitter,prompt_starter,low_engagement_flag,early_start_score,frequency_score,intensity_score,forum_per_login,completion_x_frequency,procrastination_score,engagement_breadth,registration_delay_days,registered_before_start,vle_total_clicks,vle_active_days,vle_distinct_sites,log_vle_total_clicks,log_vle_active_days,log_vle_distinct_sites,clicks_per_active_day,log_clicks_per_active_day,vle_activity_type_diversity,vle_clicks_forumng_ratio,vle_clicks_resource_ratio,vle_clicks_oucontent_ratio,vle_clicks_quiz_ratio,vle_clicks_homepage_ratio,vle_clicks_subpage_ratio,vle_clicks_url_ratio,assess_n_submitted,assess_mean_score,assess_n_banked,assess_mean_days_before_deadline,assess_n_late,assessment_completion_rate,late_assessment_rate,assignment_timeliness_score,login_time_Afternoon,login_time_Evening,login_time_Morning,login_time_Night,final_result,final_result_encoded,passed
0,11391,AAA,2013J,2,0.934,5,9.0,-0.2,0,0,0,0,0,0.100000,1.239393,1.537005,1.666667,1.868,-0.597998,4,-159.0,1,934.0,40.0,55.0,6.840547,3.713572,4.025352,23.350000,3.192532,6,0.206638,0.013919,0.592077,0.0,0.147752,0.034261,0.005353,5.0,82.0,0.0,1.8,0.0,1.0,0.0,-0.319689,1,0,0,0,Pass,2,1
1,28400,AAA,2013J,6,0.879,0,2.0,-1.5,0,0,0,1,0,0.333333,0.394069,1.794023,0.000000,5.274,-1.242339,3,-53.0,1,1435.0,80.0,84.0,7.269617,4.394449,4.442651,17.937500,2.941144,7,0.290592,0.008362,0.374216,0.0,0.225784,0.060627,0.033449,5.0,66.4,0.0,0.0,2.0,1.0,0.4,-0.394787,0,1,0,0,Pass,2,1
2,30268,AAA,2013J,0,0.522,0,90.0,3.4,1,1,0,0,0,0.010989,-1.330395,-0.742471,0.000000,0.000,3.250662,1,-92.0,1,281.0,12.0,22.0,5.641907,2.564949,3.135494,23.416667,3.195266,6,0.448399,0.014235,0.234875,0.0,0.209964,0.078292,0.014235,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.394787,0,1,0,0,Withdrawn,0,0
3,31604,AAA,2013J,8,0.521,2,3.0,2.2,0,0,0,1,0,0.250000,1.766877,-0.366957,0.222222,4.168,-0.042435,3,-52.0,1,2158.0,123.0,82.0,7.677400,4.820282,4.418841,17.544715,2.920185,8,0.293791,0.008804,0.387396,0.0,0.200185,0.066728,0.041705,5.0,76.0,0.0,2.0,0.0,1.0,0.0,-0.311345,0,1,0,0,Pass,2,1
4,32885,AAA,2013J,0,0.240,2,7.0,-0.4,0,0,0,1,0,0.125000,-0.532409,-0.327985,2.000000,0.000,-0.728053,2,-176.0,1,1034.0,70.0,66.0,6.942157,4.262680,4.204693,14.771429,2.758200,7,0.187621,0.043520,0.477756,0.0,0.197292,0.076402,0.013540,5.0,54.4,0.0,-11.4,5.0,1.0,1.0,-0.870410,1,0,0,0,Pass,2,1


## 2. Reconcile only dependent engineered features

The source cleaning decisions remain unchanged. The notebook:

- recomputes z-score and interaction features affected by capped `login_weekly` or `forum_posts_count`;
- recomputes log features after capped VLE base columns;
- reconstructs `clicks_per_active_day` from the cleaned numerator and denominator, while preserving the already agreed upper cap;
- refreshes dependent engagement flags using the reconciled dataset distribution.

No rows are removed, and no unrelated column is edited.


In [3]:
def safe_zscore(series):
    series = pd.Series(series, index=series.index, dtype=float)
    std = float(series.std(ddof=1))
    if not np.isfinite(std) or std == 0:
        return pd.Series(np.zeros(len(series)), index=series.index, dtype=float)
    return (series - float(series.mean())) / std


reconciled_df = source_df.copy()

# Preserve the upper cap already present in the agreed recleaned file.
clicks_per_active_day_cap = float(source_df["clicks_per_active_day"].max())

reconciled_df["frequency_score"] = (
    safe_zscore(reconciled_df["login_weekly"])
    + safe_zscore(reconciled_df["forum_posts_count"])
)
reconciled_df["forum_per_login"] = (
    reconciled_df["forum_posts_count"]
    / (reconciled_df["login_weekly"] + 1)
)
reconciled_df["completion_x_frequency"] = (
    reconciled_df["video_completion_rate"]
    * reconciled_df["login_weekly"]
)

reconciled_df["low_engagement_flag"] = (
    (reconciled_df["login_weekly"] <= reconciled_df["login_weekly"].quantile(0.25))
    & (
        reconciled_df["video_completion_rate"]
        <= reconciled_df["video_completion_rate"].quantile(0.25)
    )
).astype(int)
reconciled_df["engagement_breadth"] = (
    (reconciled_df["login_weekly"] > reconciled_df["login_weekly"].median()).astype(int)
    + (
        reconciled_df["video_completion_rate"]
        > reconciled_df["video_completion_rate"].median()
    ).astype(int)
    + (
        reconciled_df["forum_posts_count"]
        > reconciled_df["forum_posts_count"].median()
    ).astype(int)
    + (
        reconciled_df["submission_timeliness_days"]
        < reconciled_df["submission_timeliness_days"].median()
    ).astype(int)
)

reconciled_df["log_vle_total_clicks"] = np.log1p(
    reconciled_df["vle_total_clicks"]
)
reconciled_df["log_vle_active_days"] = np.log1p(
    reconciled_df["vle_active_days"]
)
reconciled_df["log_vle_distinct_sites"] = np.log1p(
    reconciled_df["vle_distinct_sites"]
)

raw_clicks_per_active_day = np.divide(
    reconciled_df["vle_total_clicks"],
    reconciled_df["vle_active_days"],
    out=np.zeros(len(reconciled_df), dtype=float),
    where=reconciled_df["vle_active_days"].to_numpy(dtype=float) != 0,
)
reconciled_df["clicks_per_active_day"] = np.minimum(
    raw_clicks_per_active_day,
    clicks_per_active_day_cap,
)
reconciled_df["log_clicks_per_active_day"] = np.log1p(
    reconciled_df["clicks_per_active_day"]
)

RECOMPUTED_FEATURES = [
    "frequency_score",
    "forum_per_login",
    "completion_x_frequency",
    "low_engagement_flag",
    "engagement_breadth",
    "log_vle_total_clicks",
    "log_vle_active_days",
    "log_vle_distinct_sites",
    "clicks_per_active_day",
    "log_clicks_per_active_day",
]

print("Recomputed features:", RECOMPUTED_FEATURES)
print("Preserved clicks_per_active_day cap:", clicks_per_active_day_cap)


Recomputed features: ['frequency_score', 'forum_per_login', 'completion_x_frequency', 'low_engagement_flag', 'engagement_breadth', 'log_vle_total_clicks', 'log_vle_active_days', 'log_vle_distinct_sites', 'clicks_per_active_day', 'log_clicks_per_active_day']
Preserved clicks_per_active_day cap: 55.56148837209295


## 3. Reconciliation audit

The audit verifies formula invariants and confirms that every non-recomputed column remains byte-for-byte equal in the in-memory table.


In [4]:
audit_rows = []
for column in RECOMPUTED_FEATURES:
    old_values = source_df[column]
    new_values = reconciled_df[column]
    difference = np.abs(
        old_values.to_numpy(dtype=float) - new_values.to_numpy(dtype=float)
    )
    audit_rows.append({
        "feature": column,
        "rows_changed": int(np.count_nonzero(difference > 1e-12)),
        "change_pct": float(np.mean(difference > 1e-12) * 100),
        "mean_abs_change": float(difference.mean()),
        "max_abs_change": float(difference.max()),
    })

reconciliation_audit = pd.DataFrame(audit_rows)

unchanged_columns = [
    column for column in source_df.columns
    if column not in RECOMPUTED_FEATURES
]
pd.testing.assert_frame_equal(
    source_df[unchanged_columns],
    reconciled_df[unchanged_columns],
    check_dtype=True,
    check_exact=True,
)

expected = {
    "frequency_score": (
        safe_zscore(reconciled_df["login_weekly"])
        + safe_zscore(reconciled_df["forum_posts_count"])
    ),
    "forum_per_login": (
        reconciled_df["forum_posts_count"]
        / (reconciled_df["login_weekly"] + 1)
    ),
    "completion_x_frequency": (
        reconciled_df["video_completion_rate"]
        * reconciled_df["login_weekly"]
    ),
    "log_vle_total_clicks": np.log1p(reconciled_df["vle_total_clicks"]),
    "log_vle_active_days": np.log1p(reconciled_df["vle_active_days"]),
    "log_vle_distinct_sites": np.log1p(reconciled_df["vle_distinct_sites"]),
    "log_clicks_per_active_day": np.log1p(
        reconciled_df["clicks_per_active_day"]
    ),
}

formula_errors = []
for feature, expected_values in expected.items():
    max_error = float(
        np.max(
            np.abs(
                reconciled_df[feature].to_numpy(dtype=float)
                - np.asarray(expected_values, dtype=float)
            )
        )
    )
    formula_errors.append({"feature": feature, "max_abs_formula_error": max_error})
    if max_error > 1e-10:
        raise ValueError(f"{feature} failed formula validation: {max_error}")

if reconciled_df.duplicated(KEY_COLUMNS).any():
    raise ValueError("Reconciliation introduced duplicate keys")
if len(reconciled_df) != len(source_df):
    raise ValueError("Reconciliation changed the row count")
if list(reconciled_df.columns) != list(source_df.columns):
    raise ValueError("Reconciliation changed the column order")
if not np.isfinite(
    reconciled_df.select_dtypes(include=[np.number]).to_numpy(dtype=float)
).all():
    raise ValueError("Reconciled numeric data contain missing or infinite values")

login_time_columns = [
    column for column in reconciled_df.columns
    if column.startswith("login_time_")
]
if login_time_columns and not reconciled_df[login_time_columns].sum(axis=1).eq(1).all():
    raise ValueError("Login-time one-hot columns do not sum to one")

vle_ratio_columns = [
    column for column in reconciled_df.columns
    if column.startswith("vle_clicks_") and column.endswith("_ratio")
]
if vle_ratio_columns and (
    reconciled_df[vle_ratio_columns].sum(axis=1) > 1 + 1e-10
).any():
    raise ValueError("VLE ratios sum to more than one")

formula_audit = pd.DataFrame(formula_errors)

display(reconciliation_audit.round(8))
display(formula_audit)
print("All reconciliation checks passed.")


,feature,rows_changed,change_pct,mean_abs_change,max_abs_change
0,frequency_score,32593,100.000000,0.035169,3.465415
1,forum_per_login,415,1.273280,0.001730,1.000000
2,completion_x_frequency,230,0.705673,0.015692,9.370000
3,low_engagement_flag,0,0.000000,0.000000,0.000000
4,engagement_breadth,0,0.000000,0.000000,0.000000
5,log_vle_total_clicks,326,1.000215,0.002557,1.131538
6,log_vle_active_days,315,0.966465,0.000831,0.247836
7,log_vle_distinct_sites,319,0.978738,0.000985,0.528698
8,clicks_per_active_day,473,1.451232,0.085345,20.650995
9,log_clicks_per_active_day,735,2.255085,0.004218,1.368250


,feature,max_abs_formula_error
0,frequency_score,0.0
1,forum_per_login,0.0
2,completion_x_frequency,0.0
3,log_vle_total_clicks,0.0
4,log_vle_active_days,0.0
5,log_vle_distinct_sites,0.0
6,log_clicks_per_active_day,0.0


All reconciliation checks passed.


## 4. Save the Kaggle handoff

The original source file is not overwritten. Model development must attach this notebook's saved output and load only `data_reconciled_after_fe.csv` plus its metadata.


In [5]:
winsorization_columns = [
    "login_weekly",
    "forum_posts_count",
    "vle_total_clicks",
    "vle_active_days",
    "vle_distinct_sites",
    "clicks_per_active_day",
    "assess_mean_score",
]
winsorization_caps = {
    column: float(source_df[column].max())
    for column in winsorization_columns
    if column in source_df.columns
}

zscore_columns = [
    "login_weekly",
    "forum_posts_count",
    "video_completion_rate",
    "submission_timeliness_days",
    "days_to_first_filled",
]
preprocessing_state = {
    "days_to_first_median": float(reconciled_df["days_to_first_filled"].median()),
    "login_q25": float(reconciled_df["login_weekly"].quantile(0.25)),
    "video_q25": float(reconciled_df["video_completion_rate"].quantile(0.25)),
    "login_median": float(reconciled_df["login_weekly"].median()),
    "video_median": float(reconciled_df["video_completion_rate"].median()),
    "forum_median": float(reconciled_df["forum_posts_count"].median()),
    "timeliness_median": float(
        reconciled_df["submission_timeliness_days"].median()
    ),
    "zscore": {
        column: {
            "mean": float(reconciled_df[column].mean()),
            "std": float(reconciled_df[column].std(ddof=1)),
        }
        for column in zscore_columns
    },
}

assessment_max_submitted = {
    f"{module}|{presentation}": float(value)
    for (module, presentation), value in (
        reconciled_df.groupby(
            ["code_module", "code_presentation"]
        )["assess_n_submitted"].max().items()
    )
}

fractional_count_candidates = [
    "login_weekly",
    "forum_posts_count",
    "vle_total_clicks",
    "vle_active_days",
    "vle_distinct_sites",
    "assess_n_submitted",
    "assess_n_banked",
    "assess_n_late",
]
fractional_count_features = [
    column
    for column in fractional_count_candidates
    if column in reconciled_df.columns
    and not np.allclose(
        reconciled_df[column],
        np.round(reconciled_df[column]),
        atol=1e-10,
        rtol=0,
    )
]

output_csv = ARTIFACT_DIR / "data_reconciled_after_fe.csv"
metadata_path = ARTIFACT_DIR / "reconciliation_metadata.json"
audit_path = REPORT_DIR / "reconciliation_audit.csv"
formula_path = REPORT_DIR / "formula_audit.csv"

reconciled_df.to_csv(output_csv, index=False)
reconciliation_audit.to_csv(audit_path, index=False)
formula_audit.to_csv(formula_path, index=False)

metadata = {
    "source": {
        "filename": source_path.name,
        "path": str(source_path),
        "sha256": source_sha256,
        "rows": int(len(source_df)),
        "columns": int(source_df.shape[1]),
    },
    "output": {
        "filename": output_csv.name,
        "rows": int(len(reconciled_df)),
        "columns": int(reconciled_df.shape[1]),
        "key_columns": KEY_COLUMNS,
    },
    "reconciliation": {
        "recomputed_features": RECOMPUTED_FEATURES,
        "unchanged_columns_preserved_exactly": True,
        "row_count_preserved": True,
        "column_order_preserved": True,
        "clicks_per_active_day_cap": clicks_per_active_day_cap,
    },
    "winsorization_caps": winsorization_caps,
    "fractional_count_features": fractional_count_features,
    "preprocessing_state": preprocessing_state,
    "assessment_max_submitted": assessment_max_submitted,
    "outcome_columns_for_profiling_only": [
        "final_result",
        "final_result_encoded",
        "passed",
    ],
    "excluded_from_clustering": [
        "final_result",
        "final_result_encoded",
        "passed",
    ],
}

with metadata_path.open("w", encoding="utf-8") as file:
    json.dump(metadata, file, indent=2)

print("Saved outputs:")
for path in sorted(OUTPUT_ROOT.rglob("*")):
    if path.is_file():
        print(" -", path)


Saved outputs:
 - /kaggle/working/data_reconcile/artifacts/data_reconciled_after_fe.csv
 - /kaggle/working/data_reconcile/artifacts/reconciliation_metadata.json
 - /kaggle/working/data_reconcile/reports/formula_audit.csv
 - /kaggle/working/data_reconcile/reports/reconciliation_audit.csv


## Run result required before continuing

A successful run must end with:

- `All reconciliation checks passed.`
- `data_reconciled_after_fe.csv`
- `reconciliation_metadata.json`
- `reconciliation_audit.csv`
- `formula_audit.csv`

Save a Kaggle version of this notebook, then attach that notebook output—without also attaching an older model output—to `model_development.ipynb`.
